In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/katla@mail.bradley.edu/Consolidated_Pipeline/SETUP/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")


In [0]:
catalog= dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path=f's3://games-dp/{data_source}/*.csv'
print(base_path)

In [0]:
df=(spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
)
display(df)

In [0]:
df.printSchema()

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

In [0]:
df_bronze= spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")
df_bronze.show(10)

In [0]:
df_bronze.printSchema()

In [0]:
df_duplicates= df_bronze.groupBy("customer_id").count().filter("count > 1")
display(df_duplicates)

In [0]:
print("count before drop: ", df_bronze.count())
df_silver=df_bronze.dropDuplicates(["customer_id"])
print("count after drop: ", df_silver.count())

In [0]:
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

In [0]:
df_silver=df_silver.withColumn(
    "customer_name",
    F.trim(F.col("customer_name"))
)

In [0]:
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

In [0]:
df_silver.select("city").distinct().display()

In [0]:
city_mapping={
    "Bengalore":"Bengaluru",
    "Bengaluruu":"Bengaluru",

    "Hyderabadd":"Hyderabad",
    "Hyderbad": "Hyderabad",

    "NewDheli":"New Delhi",
    "NewDelhi":"New Delhi",
    "NewDelhee":"New Delhi"

}

allowed=["Bengaluru", "Hyderabad", "New Delhi"]
df_silver=(
    df_silver.replace(city_mapping, subset=["city"])
    .withColumn("city",
                F.when(F.col("city").isNull(), None)
                .when(F.col("city").isin(allowed),F.col("city"))
                .otherwise("None")
                )

)

In [0]:
df_silver.select("city").distinct().show()

In [0]:
df_silver.select('customer_name').distinct().display()

In [0]:
df_silver= df_silver.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(),None)
    .otherwise(F.initcap("customer_name"))
)
df_silver.select('customer_name').distinct().display()

In [0]:
df_silver.filter(F.col("city").isNull()).show(truncate=False)

In [0]:
null_customer_names=['Sprintx Nutrition','Zenathlete Foods', 'Primefuel Nutrition',  'Recovery Lane']
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)


In [0]:
customer_city_fix={
    #sprintx
    789403: "New Delhi",

    #zenathlete
    789420: "Bengaluru",


    #Primefuel
    789521: "Hyderabad",

    #recovery lane
    789603: "Hyderabad"
}

df_fix=spark.createDataFrame(
    [(k,v) for k,v in customer_city_fix.items()],
    ["customer_id","fixed_city"]
)
display(df_fix)

In [0]:
df_silver = (
    df_silver
    .join(
        df_fix,
        "customer_id",
        "left"
    )
    .withColumn(
        "city",
        F.coalesce("city",
            "fixed_city")
    )
    .drop("fixed_city")
)
display(df_silver)

In [0]:
null_customer_names=['Sprintx Nutrition','Zenathlete Foods', 'Primefuel Nutrition',  'Recovery Lane']
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

In [0]:
df_silver = (
    df_silver
    .withColumn(
        "customer",
        F.concat_ws(
            F.lit("-"),  # separator as string
            F.col("customer_name"),
            F.coalesce(F.col("city"), F.lit("Unknown"))
        )
    )
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)
display(df_silver.limit(5))

In [0]:
df_silver.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .option("mergeSchema", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

In [0]:
df_silver=spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")

#take req cols
df_gold= df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")



In [0]:
df_gold.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
delta_table=DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_child_customers=spark.table("fmcg.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()